In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"] # 各 8530， 1066 条数据

model_id = "bert-base-uncased"

# 加载掩码语言建模(MLM) 模型
model = AutoModelForMaskedLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

# 对数据进行分词处理，并且移除标签
tokenized_train = train_data.map(preprocess_function, batched=True).remove_columns("label")
tokenized_test = test_data.map(preprocess_function, batched=True).remove_columns("label")

# 使用 Token 为单位的掩码操作，遮住 15% 的 Token，来训练模型预测被遮住的 Token
# 也可以采用 DataCollatorForWholeWordMask 遮盖单词，让训练模型预测被遮盖的单词
# 预测单词比预测 Token 更难，因为模型的处理单位是 Token, 一个单词可能由多个 Token 组成
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

# 后面的训练过程就和前面的一样了
training_args = TrainingArguments(
    "mlm_continued_pretraiing_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

tokenizer.save_pretrained("mlm_continued_pretraining_model")

trainer.train()

trainer.save_model()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
500,2.515200
1000,2.394400
1500,2.305500
2000,2.166700
2500,2.113000
3000,2.106800
3500,2.044900
4000,2.037500
4500,2.000900
5000,2.024500


In [3]:
trainer.evaluate()

{'eval_loss': 2.1256632804870605,
 'eval_runtime': 0.9141,
 'eval_samples_per_second': 1166.112,
 'eval_steps_per_second': 73.292,
 'epoch': 10.0}

In [7]:
from transformers import pipeline

def predict_mask(model_id):
    mask_filter = pipeline("fill-mask", model=model_id)
    preds = mask_filter("What a horrible [MASK]!")
    print(f"Predictions for {model_id}:")
    for pred in preds:
        print(f">>> {pred["sequence"]}")

predict_mask("bert-base-cased")
print("*" * 50)
predict_mask("mlm_continued_pretraining_model")

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Predictions for bert-base-cased:
>>> What a horrible idea!
>>> What a horrible dream!
>>> What a horrible thing!
>>> What a horrible day!
>>> What a horrible thought!
**************************************************
Predictions for mlm_continuted_pretrainning_model:
>>> what a horrible movie!
>>> what a horrible film!
>>> what a horrible thing!
>>> what a horrible mess!
>>> what a horrible idea!
